In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
transactions = pd.read_csv("../data/cleaned/cleaned_transactions.csv")
products = pd.read_csv("../data/cleaned/cleaned_products.csv")

print(transactions.shape)
print(products.shape)

(1127609, 35)
(2004, 11)


In [3]:
customers = transactions[
    [
        "customer_id",
        "customer_city",
        "customer_state",
        "customer_age_group",
        "customer_spending_tier"
    ]
].drop_duplicates()

customers.head()

,customer_id,customer_city,customer_state,customer_age_group,customer_spending_tier
0,CUST_2015_00003884,Mumbai,Maharashtra,46-55,Premium
1,CUST_2015_00011709,Allahabad,Uttar Pradesh,26-35,Standard
2,CUST_2015_00004782,Mumbai,Maharashtra,26-35,Premium
3,CUST_2015_00008105,Kolkata,West Bengal,36-45,Budget
4,CUST_2015_00002955,Ludhiana,Punjab,18-25,Standard


In [4]:
date_dim = pd.DataFrame()

date_dim["order_date"] = pd.to_datetime(
    transactions["order_date"]
).drop_duplicates()

date_dim["year"] = date_dim["order_date"].dt.year
date_dim["month"] = date_dim["order_date"].dt.month
date_dim["quarter"] = date_dim["order_date"].dt.quarter
date_dim["day"] = date_dim["order_date"].dt.day

date_dim.head()

,order_date,year,month,quarter,day
0,2015-01-25,2015.0,1.0,1.0,25.0
1,2015-01-05,2015.0,1.0,1.0,5.0
2,2015-01-24,2015.0,1.0,1.0,24.0
3,2015-01-28,2015.0,1.0,1.0,28.0
4,2015-01-31,2015.0,1.0,1.0,31.0


In [5]:
customers.to_csv("../data/cleaned/dim_customers.csv", index=False)
date_dim.to_csv("../data/cleaned/dim_date.csv", index=False)

print("Dimension files saved successfully")

Dimension files saved successfully


In [6]:
os.makedirs("../database", exist_ok=True)

conn = sqlite3.connect("../database/amazon_india.db")

In [7]:
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
date_dim.to_sql("time_dimension", conn, if_exists="replace", index=False)

print("All tables loaded into SQLite database successfully")

All tables loaded into SQLite database successfully


In [8]:
query = """
SELECT 
    order_year,
    SUM(final_amount_inr) AS total_revenue
FROM transactions
GROUP BY order_year
ORDER BY order_year;
"""

pd.read_sql(query, conn)

,order_year,total_revenue
0,2015,2.142163e+09
1,2016,3.598316e+09
2,2017,5.510026e+09
3,2018,7.248545e+09
4,2019,8.605901e+09
5,2020,1.187319e+10
6,2021,1.099021e+10
7,2022,8.532312e+09
8,2023,7.712999e+09
9,2024,6.823413e+09
